# Python Functions — Interactive Notebook

Hands-on exploration of function internals, closures, scope, recursion, and performance.

## 1. Function Attributes

In [ ]:
def greet(name: str, greeting: str = "Hello") -> str:
    """
    Return a greeting string.

    Args:
        name: The person's name.
        greeting: The greeting word.

    Returns:
        A formatted greeting string.
    """
    return f"{greeting}, {name}!"

print(f"__name__:        {greet.__name__}")
print(f"__qualname__:    {greet.__qualname__}")
print(f"__doc__:         {greet.__doc__[:40]}...")
print(f"__annotations__: {greet.__annotations__}")
print(f"__defaults__:    {greet.__defaults__}")
print(f"__module__:      {greet.__module__}")
print()
print("Code object attributes:")
code = greet.__code__
print(f"  co_varnames:   {code.co_varnames}")
print(f"  co_argcount:   {code.co_argcount}")
print(f"  co_filename:   {code.co_filename}")
print(f"  co_firstlineno:{code.co_firstlineno}")

## 2. *args and **kwargs

In [ ]:
def show_args(*args, **kwargs):
    print(f"args   = {args}   (type: {type(args).__name__})")
    print(f"kwargs = {kwargs} (type: {type(kwargs).__name__})")

print("Call 1: show_args(1, 2, 3, x=10, y=20)")
show_args(1, 2, 3, x=10, y=20)
print()

# Unpacking
nums = [1, 2, 3]
opts = {"x": 10, "y": 20}
print("Call 2: show_args(*nums, **opts)")
show_args(*nums, **opts)
print()

# Practical: pass-through wrapper
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"  Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"  Returned: {result}")
        return result
    return wrapper

@logged
def add(a, b):
    return a + b

print("Logged function call:")
add(3, 4)

## 3. Positional-Only and Keyword-Only Parameters

In [ ]:
import sys

# Keyword-only (works in all Python 3)
def connect(host, port, *, timeout=30, retries=3):
    print(f"Connecting to {host}:{port} (timeout={timeout}, retries={retries})")

connect("localhost", 8080)
connect("localhost", 8080, timeout=60)
# connect("localhost", 8080, 60)  # would raise TypeError

print()

# Positional-only (Python 3.8+)
if sys.version_info >= (3, 8):
    exec("""
def pow_pos_only(base, exp, /, mod=None):
    result = base ** exp
    return result if mod is None else result % mod

print(f"pow_pos_only(2, 10) = {pow_pos_only(2, 10)}")
print(f"pow_pos_only(2, 10, mod=7) = {pow_pos_only(2, 10, mod=7)}")
try:
    pow_pos_only(base=2, exp=10)
except TypeError as e:
    print(f"TypeError: {e}")
""")
else:
    print("Positional-only parameters require Python 3.8+")

## 4. Mutable Default Argument Bug

In [ ]:
print("=== THE BUG ===")
def buggy_append(item, lst=[]):
    lst.append(item)
    return lst

print(f"Call 1: {buggy_append(1)}")
print(f"Call 2: {buggy_append(2)}")
print(f"Call 3: {buggy_append(3)}")
print(f"Default is: {buggy_append.__defaults__}")

print()
print("=== THE FIX ===")
def fixed_append(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(f"Call 1: {fixed_append(1)}")
print(f"Call 2: {fixed_append(2)}")
print(f"Call 3: {fixed_append(3)}")

print()
print("=== INTENTIONAL USE: memoization cache ===")
def fib_cached(n, _cache={0: 0, 1: 1}):
    if n not in _cache:
        _cache[n] = fib_cached(n-1) + fib_cached(n-2)
    return _cache[n]

print([fib_cached(i) for i in range(10)])

## 5. LEGB Scope Rule

In [ ]:
x = "GLOBAL"

def outer():
    x = "ENCLOSING"

    def inner():
        x = "LOCAL"
        print(f"  inner sees: x = {x!r}")

    def inner_no_local():
        print(f"  inner_no_local sees: x = {x!r}")

    inner()
    inner_no_local()
    print(f"  outer sees: x = {x!r}")

outer()
print(f"global sees: x = {x!r}")

print()
print("=== Built-in scope ===")
print(f"len is built-in: {len([1,2,3])}")
# Shadowing built-in (don't do this!)
len_backup = len
len = lambda x: "shadowed!"
print(f"After shadowing: len([1,2,3]) = {len([1,2,3])}")
len = len_backup  # restore
print(f"After restore: len([1,2,3]) = {len([1,2,3])}")

## 6. Closures and nonlocal

In [ ]:
def make_counter(start=0):
    count = start

    def increment(step=1):
        nonlocal count
        count += step
        return count

    def reset():
        nonlocal count
        count = start

    def get():
        return count

    return increment, reset, get

inc, reset, get = make_counter(10)
print(f"Initial: {get()}")
print(f"After inc(): {inc()}")
print(f"After inc(5): {inc(5)}")
reset()
print(f"After reset(): {get()}")

print()
print("=== Closure cell contents ===")
def make_adder(n):
    def add(x):
        return x + n
    return add

add5 = make_adder(5)
print(f"add5(3) = {add5(3)}")
print(f"Closure cells: {add5.__closure__}")
print(f"Cell contents: {add5.__closure__[0].cell_contents}")

## 7. Recursion and Stack Frames

In [ ]:
import sys

print(f"Default recursion limit: {sys.getrecursionlimit()}")

# Visualize stack depth
def factorial_verbose(n, depth=0):
    indent = "  " * depth
    print(f"{indent}factorial({n}) called")
    if n <= 1:
        print(f"{indent}returning 1")
        return 1
    result = n * factorial_verbose(n - 1, depth + 1)
    print(f"{indent}returning {result}")
    return result

print()
print("Stack trace for factorial(4):")
print(f"Result: {factorial_verbose(4)}")

In [ ]:
from functools import lru_cache
import timeit

# Without memoization
def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n-1) + fib_slow(n-2)

# With lru_cache
@lru_cache(maxsize=None)
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n-1) + fib_fast(n-2)

# Iterative (best for large n)
def fib_iter(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

n = 30
t_slow = timeit.timeit(lambda: fib_slow(n), number=10)
t_fast = timeit.timeit(lambda: fib_fast(n), number=10_000)
t_iter = timeit.timeit(lambda: fib_iter(n), number=10_000)

print(f"fib({n}) = {fib_iter(n)}")
print(f"Recursive (no cache), 10 calls:    {t_slow:.3f}s")
print(f"Recursive (lru_cache), 10k calls:  {t_fast:.3f}s")
print(f"Iterative, 10k calls:              {t_iter:.3f}s")
print()
print(f"Cache info: {fib_fast.cache_info()}")

## 8. Lambda and Higher-Order Functions

In [ ]:
from functools import reduce, partial

data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]

print("=== map ===")
squares = list(map(lambda x: x**2, data))
print(f"Squares: {squares}")

print()
print("=== filter ===")
evens = list(filter(lambda x: x % 2 == 0, data))
print(f"Evens: {evens}")

print()
print("=== sorted with key ===")
words = ['banana', 'apple', 'cherry', 'date']
by_length = sorted(words, key=lambda w: len(w))
by_last   = sorted(words, key=lambda w: w[-1])
print(f"By length: {by_length}")
print(f"By last char: {by_last}")

print()
print("=== reduce ===")
product = reduce(lambda acc, x: acc * x, data)
print(f"Product of {data} = {product}")

print()
print("=== partial ===")
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube   = partial(power, exp=3)
print(f"square(5) = {square(5)}")
print(f"cube(3)   = {cube(3)}")

## 9. functools.wraps Demo

In [ ]:
from functools import wraps
import time

# Without @wraps — loses identity
def timer_bad(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  {func.__name__} took {elapsed*1000:.3f}ms")
        return result
    return wrapper

# With @wraps — preserves identity
def timer_good(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  {func.__name__} took {elapsed*1000:.3f}ms")
        return result
    return wrapper

@timer_bad
def compute_bad(n):
    """Compute sum of squares."""
    return sum(x**2 for x in range(n))

@timer_good
def compute_good(n):
    """Compute sum of squares."""
    return sum(x**2 for x in range(n))

compute_bad(10000)
compute_good(10000)

print()
print(f"Without @wraps: __name__ = {compute_bad.__name__!r}")
print(f"With    @wraps: __name__ = {compute_good.__name__!r}")
print(f"Without @wraps: __doc__  = {compute_bad.__doc__!r}")
print(f"With    @wraps: __doc__  = {compute_good.__doc__!r}")

## 10. Interview Q Demos

In [ ]:
# Q1: Mutable default bug
print("Q1: Mutable default argument")
def f(lst=[]):
    lst.append(1)
    return lst

print(f"  f() = {f()}")
print(f"  f() = {f()}")
print(f"  f() = {f()}")
print(f"  f.__defaults__ = {f.__defaults__}  (same list object!)")

In [ ]:
# Q4: Closure internals
print("Q4: Closure captures by reference, not value")

# Classic closure bug
funcs = []
for i in range(5):
    funcs.append(lambda: i)  # captures 'i' by reference!

print("  Bug: all lambdas return same value")
print(f"  [f() for f in funcs] = {[f() for f in funcs]}")

# Fix: capture by value using default argument
funcs_fixed = []
for i in range(5):
    funcs_fixed.append(lambda i=i: i)  # default arg captures current value

print("  Fix: capture by value")
print(f"  [f() for f in funcs_fixed] = {[f() for f in funcs_fixed]}")

In [ ]:
# Q5: Tail recursion — Python doesn't optimize it
import sys
print("Q5: Tail recursion not optimized")

def factorial_tail(n, acc=1):
    if n <= 1:
        return acc
    return factorial_tail(n - 1, n * acc)  # tail call — but still creates frame!

def factorial_iter(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(f"  factorial_tail(10) = {factorial_tail(10)}")
print(f"  factorial_iter(10) = {factorial_iter(10)}")

# Show recursion limit
try:
    factorial_tail(2000)
except RecursionError:
    print(f"  RecursionError at depth ~{sys.getrecursionlimit()}")

print(f"  factorial_iter(2000) works fine: {len(str(factorial_iter(2000)))} digits")